# EDA — Credit Card Fraud Detection (ULB Dataset)

**Task IDs:** T-22  
**Purpose:** Explore the ULB credit card fraud dataset — class distribution,
Amount/Time patterns, V1–V28 summary statistics, correlation structure, and
fraud-vs-legitimate feature separation ranked by KS-statistic.

Findings are recorded in `01-data/EDA Findings.md` (Obsidian vault).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp

from src.data.load import class_balance, load_raw

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 31)

df = load_raw()
df.shape

## 1. Class Distribution

In [ ]:
bal = class_balance(df)
print(f"Total:      {bal['total']:,}")
print(f"Legitimate: {bal['legit']:,}")
print(f"Fraud:      {bal['fraud']:,}")
print(f"Fraud %:    {bal['fraud_pct']}%")
print(f"Imbalance:  1:{bal['legit'] // bal['fraud']}")

fig, ax = plt.subplots(figsize=(6, 4))
counts = df["Class"].value_counts()
ax.bar(["Legitimate", "Fraud"], counts.values, color=["#2ecc71", "#e74c3c"])
ax.set_yscale("log")
ax.set_ylabel("Count (log scale)")
ax.set_title("Class Distribution")
for i, v in enumerate(counts.values):
    ax.text(i, v * 1.5, f"{v:,}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("../reports/figures/eda/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Amount and Time Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ["Amount", "Time"]):
    for cls, label, color in [(0, "Legitimate", "#2ecc71"), (1, "Fraud", "#e74c3c")]:
        data = df.loc[df["Class"] == cls, col]
        ax.hist(data, bins=50, alpha=0.5, label=label, color=color, density=True)
    ax.set_title(f"{col} Distribution by Class")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.legend()

axes[0].set_xlim(0, df["Amount"].quantile(0.99))
plt.tight_layout()
plt.savefig("../reports/figures/eda/amount_time_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Amount stats (legit):  mean={df.loc[df.Class==0,'Amount'].mean():.2f}, median={df.loc[df.Class==0,'Amount'].median():.2f}")
print(f"Amount stats (fraud):  mean={df.loc[df.Class==1,'Amount'].mean():.2f}, median={df.loc[df.Class==1,'Amount'].median():.2f}")

## 3. V1–V28 Summary Statistics

In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]
desc = df[v_cols].describe().T
desc.head(10)

## 4. Correlation Heatmap

In [ ]:
corr = df[v_cols + ["Amount", "Class"]].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("../reports/figures/eda/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

class_corr = corr["Class"].drop("Class").sort_values(key=abs, ascending=False)
print("Top 10 features by |correlation with Class|:")
print(class_corr.head(10))

## 5. Fraud vs Legitimate Feature Distributions (KS-Statistic Ranking)

The Kolmogorov-Smirnov statistic measures the maximum separation between two
cumulative distributions. A higher KS value means the feature separates fraud
from legitimate more clearly.

In [ ]:
ks_results = []
for col in v_cols + ["Amount", "Time"]:
    legit = df.loc[df["Class"] == 0, col]
    fraud = df.loc[df["Class"] == 1, col]
    stat, pval = ks_2samp(legit, fraud)
    ks_results.append({"feature": col, "ks_stat": stat, "p_value": pval})

ks_df = pd.DataFrame(ks_results).sort_values("ks_stat", ascending=False).reset_index(drop=True)
ks_df.head(10)

In [ ]:
top6 = ks_df["feature"].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.ravel(), top6):
    for cls, label, color in [(0, "Legitimate", "#2ecc71"), (1, "Fraud", "#e74c3c")]:
        data = df.loc[df["Class"] == cls, col]
        ax.hist(data, bins=50, alpha=0.5, label=label, color=color, density=True)
    ks_val = ks_df.loc[ks_df["feature"] == col, "ks_stat"].values[0]
    ax.set_title(f"{col} (KS={ks_val:.3f})")
    ax.legend(fontsize=8)
plt.suptitle("Top 6 Features by KS-Statistic", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/eda/top6_ks_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Key Findings

Record these in `01-data/EDA Findings.md` (Obsidian vault) after running:

- **Class imbalance:** ~0.17% fraud, ratio ~1:578
- **Amount:** fraud transactions tend to have [fill after running]
- **Time:** [fill after running]
- **Top separating features (KS):** [fill after running]
- **Correlation:** V-features are largely uncorrelated (PCA), but several
  correlate with Class and Amount